# Small 1D CNN — fixed configuration, no inner search

This exploratory extension uses the same eligible, labelled 500 ms windows, reviewed boundaries, exclusions and participant LOSO test assignments as the feature models. The original reviewed cohort had 9,215 windows from 15 participants; new reviews may produce different counts. Inputs are their **filtered waveform samples**, not the 36 handcrafted feature columns. It therefore changes both representation and classifier.

There is one predeclared architecture/training configuration. No inner cross-validation, hyperparameter search, feature selection, early stopping or threshold tuning is performed. Each outer training fold fits its own scalar normalization and CNN. The held-out participant is evaluated once after the fixed training budget. Fifteen evaluation fits and one final fit replace thousands of grid-search fits. No real-data CNN training was run locally.

## Portable setup

This is a clean public source notebook; no participant results are embedded.
Open it from the cloned project. If necessary set `EMG_PROJECT_ROOT` to that
project root. There is no personal Drive path or automatic Drive mount. Mount
your own private storage separately when using Colab. Restore authorized private
inputs as described in the root `README.md` (Dataset and experiment annotations) before running data cells.

Install root `requirements.txt` in the selected environment, restart that kernel,
then run cells in order. The optional installer below defaults to False. CNN
users also need compatible PyTorch; see the root README. Scientific settings,
labels, grouping and model procedures are retained; setup/path handling changes.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from importlib.metadata import version

# Set this only when the notebook is opened outside the cloned project.
PROJECT_ROOT_OVERRIDE = os.environ.get('EMG_PROJECT_ROOT')
start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in [start, *start.parents]
    if (p / 'project_paths.py').is_file() and (p / 'modelling/core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the cloned project, or set EMG_PROJECT_ROOT to its root.')
sys.path.insert(0, str(PROJECT_ROOT))
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
    raise RuntimeError('Restart this kernel, restore INSTALL_REQUIREMENTS=False, then run from the first cell.')
for package, expected in {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0'}.items():
    actual=version(package)
    print(package,actual)
    if actual!=expected:raise RuntimeError('Install requirements in this environment and restart the kernel.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
probe=subprocess.run([sys.executable,'-c','import numpy, pandas, scipy.sparse, sklearn; print("Fresh-process imports OK")'],capture_output=True,text=True)
print(probe.stdout)
if probe.returncode:raise RuntimeError(probe.stderr)
print('Project:',PROJECT_ROOT)
print('Python:',sys.version.split()[0])


In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Image, Markdown
from modelling.cnn import CNN_CONFIG, SmallCNN, load_signals, run_cnn, predict_saved

DATA_DIR = PROJECT_ROOT / "modelling/cnn_data"
MODEL_DATA_DIR = PROJECT_ROOT / "modelling/data"
OUTPUT_ROOT = PROJECT_ROOT / "modelling/outputs"
CONFIG = dict(CNN_CONFIG)
signals, windows, provenance = load_signals(DATA_DIR, MODEL_DATA_DIR)
print("Original reviewed cohort: 15 subjects, 139 sets, 9215 windows. New reviews may differ.")
display(pd.DataFrame([{"subjects":windows.subject_id.nunique(),"sets":windows.recording_id.nunique(),
    "windows":len(windows),"samples_per_window":signals.shape[1],
    "trainable_parameters":sum(p.numel() for p in SmallCNN().parameters()),
    "gpu_available":torch.cuda.is_available(),"torch_version":str(torch.__version__)}]))
display(pd.Series(CONFIG,name="Fixed choices — no search").to_frame())
if not torch.cuda.is_available():
    print("CPU selected. A GPU-enabled Colab runtime may run this experiment faster.")


## 2. Labels, eligibility and class counts

Label **0**: first third of the reviewed active interval. Label **1**: last third. These are **elapsed-time thirds**, adapting the assignment's repetition-count example because repetition markers are unavailable. The middle third and any 500 ms windows crossing a stage boundary are excluded. Hop is 250 ms. Keep both legs/all sets together in every split.

The bundle honors the saved set exclusion (`S009_REC01`), reviewed boundaries and computational/flatline/manual-artifact eligibility rules. Provisional transient/drift flags remain flags rather than automatic exclusions. MVC normalization cannot be performed without MVC references. Preprocessing is offline and can use later samples within the same recording; this does not establish real-time performance.

Earlier whole-cohort statistics and trajectories informed development. Fold-local selection prevents direct fitting on held-out subjects, but does not make this a pristine confirmatory study. The exclusion and boundary reviews also preceded validation. A new cohort would provide a stronger independent test.

### Descriptive class counts

The CNN uses the same retained early/late labels as the feature models. Check counts and percentages overall, by participant and leg. Windows overlap and are not independent participants. No inner validation partitions are created for this fixed experiment.

In [ ]:
from modelling.class_balance import count_table
BALANCE_OUT = OUTPUT_ROOT / "class_balance_cnn" / provenance["source_model_data_sha256"][:12]
BALANCE_OUT.mkdir(parents=True,exist_ok=True)
for name,keys in [("overall",[]),("by_subject",["subject_id"]),("by_leg",["leg"])]:
    table=count_table(windows,keys)
    table.to_csv(BALANCE_OUT / f"class_balance_{name}.csv",index=False)
    display(Markdown("**"+name.replace("_"," ")+"**"));display(table.round(3))
print("Same early/late rows as the feature models. No inner split or model fitting in this descriptive audit.")


## 3. Fixed CNN experiment: no inner splits

**Evaluation:** LOSO over the same 15 participants. Both legs/all recordings stay together. The CNN receives only waveforms of shape `(1, 500)`; participant ID, leg, set, timestamps and position never become predictors.

**Architecture (2,785 trainable parameters):** Conv1D(1→16, kernel 9, padding 4) → ReLU → max-pool 4 → Conv1D(16→32, kernel 5, padding 2) → ReLU → max-pool 4 → global average pooling → dropout 0.2 → one output logit. Convolution operates over successive signal samples, where ordering has physical meaning. These are compact pragmatic choices, not claimed physiological constants or literature-optimized settings.

**Training:** 20 fixed epochs, batch size 128, AdamW learning rate 0.001 and weight decay 0.0001, binary cross-entropy with logits. Mini-batches shuffle training windows only; optimization weights windows equally. No class weighting, augmentation, resampling or hyperparameter trials. Do not choose a best epoch using held-out results.

**Normalization:** fit one mean and one SD over the waveform samples in the outer training set, then freeze them for the held-out participant. This is neither per-window amplitude normalization nor MVC normalization. It preserves amplitude differences relative to the training scale. Windows are already finite by upstream eligibility checks; no waveform imputation is fitted.

**Predictions:** score is the raw uncalibrated output logit; predict class 1 only for a logit >0. No threshold tuning or probability calibration. ROC/PR use those scores within each held-out participant. All model/normalization fitting is confined to the outer training participants.

**Reproducibility:** base seed 20261018; fold seed = base+fold, final seed=base+1000. Seed Python, NumPy, PyTorch and GPU generators; deterministic PyTorch algorithms are requested. Data loading uses zero worker processes; numerical CPU threads are limited to one. Library versions, device, source/input hashes and settings are saved. Identical results across different devices/library versions are not guaranteed.

There is no inner selection because settings are fixed beforehand. The final all-participant model uses the same 20 epochs and gives no new independent test score. This CNN was proposed after viewing earlier models, so it is an exploratory extension; changing settings after seeing its test results would add further development on the same cohort.

## 4. Run one fixed CNN per held-out participant

The next cell performs the real training in your runtime. It prints the device, fold start, held-out subject, final fold macro-F1 and duration. It saves the model/normalization together, exact split, training-only loss history and partial results after each fold. No test metric is computed during epoch selection because there is no epoch selection. New runs use new timestamped directories and preserve previous model outputs. Runtime still depends on the device; fewer fits do not guarantee a specific completion time.

In [ ]:
OUT = run_cnn(DATA_DIR, OUTPUT_ROOT, CONFIG, model_data_dir=MODEL_DATA_DIR)
print("Completed run:", OUT)
run_manifest=json.loads((OUT / "run_manifest.json").read_text())
assert run_manifest["status"]=="complete"
assert run_manifest["n_outer_folds"]==windows.subject_id.nunique()


## 5. Required metrics for Sections 5–6

Report accuracy; macro precision, recall and F1; F1 per class; Cohen's kappa; ROC-AUC; trapezoidal PR-AUC; and average precision. Summaries are unweighted mean ± sample SD across the 15 held-out participants (`ddof=1`); SD describes participant variability, not a confidence interval. Per-class tables include supports. Undefined AUCs remain missing with defined-fold counts.

Late stage is positive class 1. CNN logits are uncalibrated scores, not probabilities. Evaluate AUCs within each subject rather than pooling scores from separately trained models. PR-AUC and average precision use different integration conventions. Pooled confusion counts and pooled classification metrics are secondary and give more weight to subjects with more windows.

Early/late elapsed-time labels remain stage proxies, not independent evidence of peripheral fatigue. The training-loss plot is a diagnostic; do not select epochs or architectures using these outer results.

In [ ]:
fold_metrics = pd.read_csv(OUT / "results.csv")
summary = pd.read_csv(OUT / "metrics_mean_sd.csv")
summary["mean_plus_minus_sd"] = summary.apply(
    lambda r: f"{r['mean']:.3f} ± {r['sd']:.3f}", axis=1)
display(summary[["metric", "mean_plus_minus_sd", "n_defined", "n_folds"]])
display(fold_metrics[["fold", "test_subject", "accuracy", "precision_macro", "recall_macro",
    "f1_macro", "f1_class_0", "f1_class_1", "cohen_kappa", "roc_auc", "pr_auc", "average_precision"]])
display(pd.read_csv(OUT / "per_class_metrics.csv"))
display(pd.read_csv(OUT / "per_class_mean_sd.csv", header=[0,1], index_col=0))
display(Markdown("**Secondary pooled window metrics:**"))
display(pd.Series(json.loads((OUT / "pooled_window_metrics_secondary.json").read_text())).to_frame("value"))

In [ ]:
for image_name in ["confusion_matrices.png", "roc_pr_by_subject.png",
                   "per_subject_f1.png", "metrics_mean_sd.png", "validation_diagram.png", "training_loss.png"]:
    display(Image(filename=str(OUT / "figures" / image_name)))

## 6. Learned representation and bilateral results

There is no handcrafted-feature selection or feature-frequency table for this CNN. Its convolution filters learn signal patterns within the training subjects. There is no direct one-to-one physiological interpretation of those filters or the 32 pooled channels. A feature coefficient table is not fabricated.

Below, show the fixed input/model size and held-out per-leg performance. Both legs were grouped by participant during evaluation. These are separate sets; simultaneous bilateral relationships cannot be inferred. The comparison with earlier feature models changes representation, classifier and training procedure, so any performance difference cannot be attributed to CNN architecture alone.

In [ ]:
display(fold_metrics[["test_subject","input_samples","trainable_parameters","seconds"]])
display(pd.read_csv(OUT / "per_subject_leg_metrics.csv"))
display(pd.read_csv(OUT / "paired_leg_f1.csv"))
oof=pd.read_csv(OUT / "out_of_fold_predictions.csv")
assert oof.window_id.is_unique and len(oof)==len(windows)
print("Exactly one held-out prediction for every eligible labelled window.")


## 7. Keep the CNN results and use the final model

Each run saves its own `outputs/cnn_fixed/<run>/results.csv`, one row per held-out participant, plus all requested tables/figures, raw scores/predictions and `run_manifest.json`. Original model results and combined tables are not overwritten.

Each fold and `final_refit/` saves **`model.pt`**, containing the learned state and training-fitted normalization. For new recordings, reproduce the same filtered 500-sample windows and use `predict_saved` in `cnn.py`; never fit normalization to the new participant. Only load your own trusted model files.

The final model is trained on all participants and has no new independent test score. Preserve the predefined settings when interpreting this run. A new cohort would be required for stronger independent confirmation after these exploratory model comparisons.

The last cell archives this model's results and descriptive class-balance tables under the local private outputs directory. Save the executed notebook too. No raw-data upload or repeated dataset upload is required once `cnn_data/` is present.

In [ ]:
import shutil
# Include the descriptive audit in this model's downloadable results.
shutil.copytree(BALANCE_OUT, OUT / "class_balance", dirs_exist_ok=True)
archive = shutil.make_archive(str(OUT.parent / (OUT.name + "_results")), "zip", root_dir=OUT)
print("Results archive:", archive)
print("Save this executed notebook separately to retain displayed outputs.")
print("CNN fold results:",OUT / "results.csv")
print("Original model aggregates are preserved.")
